# IEDA 3130 · Pose Estimation Demo (Colab version)
HKUST IEDA 3130 Ergonomics and Safety Management · TA: Junjie HOU

Use this notebook if you have no camera, no admin rights, or cannot install MediaPipe locally.

Run the cells top-to-bottom (Shift+Enter). If a cell fails, **Runtime → Restart session** and run again from cell 2.

## 0 · Install and download the code
This installs MediaPipe and clones the GitHub repository into the Colab runtime (`/content/ieda3130-pose-demo`).
It takes about 30–60 s. Re-running the cell is safe.

In [ ]:
!pip -q install "mediapipe>=0.10.14,<2"
import os
if not os.path.isdir('ieda3130-pose-demo'):
    !git clone -q https://github.com/Junjie31/ieda3130-pose-demo.git
assert os.path.isdir('ieda3130-pose-demo'), 'git clone failed - check your network and the repo URL'
%cd ieda3130-pose-demo
import mediapipe, cv2
print('mediapipe', mediapipe.__version__, '| opencv', cv2.__version__)

## 1 · Upload a photo
Side view, whole body, plain background (see README §4). If you cancel the upload, a sample fitness photo is downloaded instead so you can still try the tool.

In [ ]:
from google.colab import files
up = files.upload()
IMG = next(iter(up)) if up else None
if IMG is None:
    !wget -q -O sample.jpg https://storage.googleapis.com/mediapipe-assets/pose.jpg
    IMG = 'sample.jpg'
print('using', IMG)

## 2 · Run the demo
Add `--known shoulder_elbow=33.5` (your tape measurement in cm) to get the `calibrated_cm` column.
If both arms are visible (front view) and you measured only one arm, also add `--known-side R` (or `L`).

In [ ]:
# run on the uploaded photo (change the --known value to your own tape measurement)
!python pose_demo.py --image "$IMG" --no-show --model full

# with tape calibration from the right arm:
# !python pose_demo.py --image "$IMG" --no-show --model full --known shoulder_elbow=33.5 --known-side R

In [ ]:
import os, pandas as pd
from IPython.display import Image, display
stem = os.path.splitext(os.path.basename(IMG))[0]
display(Image(f'outputs/{stem}_pose.jpg', width=520))
print('--- measurements (' + stem + '_measures.csv) ---')
display(pd.read_csv(f'outputs/{stem}_measures.csv').fillna(''))
print('--- joint angles, degrees (' + stem + '_angles.csv) ---')
display(pd.read_csv(f'outputs/{stem}_angles.csv').fillna(''))

## 3 · Webcam snapshot (optional)
Colab cannot stream video, but it can take one photo from your browser camera. Click **Capture**, then run the demo on `webcam.jpg`.

In [ ]:
from IPython.display import Javascript
from google.colab.output import eval_js
from base64 import b64decode
js = Javascript('''
async function takePhoto() {
  const div = document.createElement('div');
  const video = document.createElement('video'); video.style.display='block';
  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  document.body.appendChild(div); div.appendChild(video); video.srcObject = stream; await video.play();
  const btn = document.createElement('button'); btn.textContent = 'Capture'; div.appendChild(btn);
  await new Promise(r => btn.onclick = r);
  const c = document.createElement('canvas'); c.width = video.videoWidth; c.height = video.videoHeight;
  c.getContext('2d').drawImage(video, 0, 0); stream.getVideoTracks()[0].stop(); div.remove();
  return c.toDataURL('image/jpeg', 0.9);
}''')
display(js)
data = eval_js('takePhoto()')
open('webcam.jpg','wb').write(b64decode(data.split(',')[1]))
!python pose_demo.py --image webcam.jpg --no-show --model full
display(Image('outputs/webcam_pose.jpg', width=520))

## 4 · Download your results
Fill in `WORKSHEET.md` with the numbers above and compare with your Lab 2 Excel sheet.

In [ ]:
!zip -q -r outputs.zip outputs
files.download('outputs.zip')

## Notes and troubleshooting

- **Re-running:** cell 0 is safe to re-run. If `%cd` ever fails, just re-run cell 0.
- **Where are my files?** `outputs/<name>_pose.jpg` (skeleton overlay) and `outputs/<name>_measures.csv` (measurement table).
- **`--known` names:** `shoulder_elbow`, `elbow_fingertip`, `shoulder_wrist`, `hip_knee`, `knee_ankle` (straight-line lengths only).
- **Webcam cell does nothing?** Your browser may block camera access — allow it in the URL-bar permission prompt.
- **Slow / out of memory?** Use the `lite` model (remove `--model full`).
- **Want the plain local commands?** See `README.md` in the cloned folder.
- **Privacy:** never upload photographs of other people without their consent. Use your own photo.